# Representation augmentation — fixing the resource-notation brittleness

E5 showed the model transfers to real tool-call *vocabulary* but is brittle
to resource *notation*: the released model scores 90.8% on tau2 in the
trained `family:namespace/leaf` notation but ~55% on the same data in an
all-colon notation. Here we retrain CE with **notation augmentation** (each
training trace re-notated under a random delimiter scheme; labels
re-verified, provably unchanged) and test whether the model then handles
real data in *varied* notations — i.e. whether the brittleness is fixable,
not fundamental.

**Data hygiene (held out properly).**
- **train** = `augmented_train` (seed 101 corpus, re-notated seed 33)
- **val** = `augmented_val` (seed 202 corpus, re-notated seed 34) — a *different*
  seed, deduplicated against train and the committed test; augmented too so
  monitoring reflects the mixed-notation distribution.
- **test**, never seen in training, from two independent real logs:
  - **tau2** (`Jarrodbarnes/tau2-sft-v4`) — balanced confused-deputy set, in
    both the trained slash notation and the naive colon notation.
  - **Toucan** (`Agent-Ark/Toucan-1.5M`) — a *second, independent* real MCP
    corpus (thousands of servers), authorized-only, rendered in **mixed native
    notations**: a recognition test (false-refuse on real legitimate calls)
    across real vocabulary *and* real formats.

Plain `!` commands, `python -u` streams live. Runtime → T4/L4 → Run all
(~1.5–2 h for 3 seeds). Download `augment_results.zip` at the end.

In [ ]:
import os
os.chdir('/content')
if not os.path.isdir('verifier-as-reward'):
    !git clone https://github.com/esmaeil-abedi-dev/verifier-as-reward.git
os.chdir('/content/verifier-as-reward')
!git pull --ff-only
import torch; assert torch.cuda.is_available()
!PYTHONPATH=. python test_authority_verifier.py
!PYTHONPATH=. python make_expanded_train.py --train-seed 101 --train-traces-per-class 150 --val-seed 202 --val-traces-per-class 25

## Step 1 — build the notation-augmented training corpus (+ tests)

In [ ]:
# augment BOTH the train (seed-101) and val (seed-202) corpora, seed-separated,
# each trace re-notated under a random delimiter scheme (labels re-verified)
!PYTHONPATH=. python -u augment_representation.py --in expanded_train.jsonl --out augmented_train.jsonl --seed 33
!PYTHONPATH=. python -u augment_representation.py --in expanded_val.jsonl   --out augmented_val.jsonl   --seed 34
!PYTHONPATH=. python test_augment_representation.py
!PYTHONPATH=. python test_map_toucan.py

## Step 2 — train CE on the augmented corpus (3 seeds; set `7` only for a quick look)

In [ ]:
!for s in 7 8 9; do echo "=== augment seed $s ==="; PYTHONPATH=. python -u train_verifier_reward.py --model Qwen/Qwen2.5-0.5B --ce-loss --balance-reward --prompt-style nl --steps 500 --batch-size 16 --lr 2e-5 --train-file augmented_train.jsonl --test-file augmented_val.jsonl --eval-every 50 --eval-max-actions 120 --seed $s --log-file training_log_augment_seed$s.jsonl --save-dir ckpt_augment_seed$s || break; done

## Step 3 — build the held-out REAL tests (never seen in training)

Two independent real logs. **tau2** (balanced confused-deputy) in both the
trained slash notation and the naive colon notation. **Toucan** (second,
independent source; authorized-only) rendered in **mixed native notations**
via `augment()` — each real trace gets a random delimiter scheme, labels
re-verified. All labels come from our verifier.

In [ ]:
from trace_benchmark import load_traces, write_jsonl
from augment_representation import renotate_trace, augment

# tau2 — balanced confused-deputy, slash (trained) + colon (naive)
!PYTHONPATH=. python map_tau_to_chain.py --seed 5   # writes real_trace_*.jsonl (slash)
slash = load_traces('real_trace_all.jsonl')
write_jsonl([renotate_trace(t, 'allcolon') for t in slash], 'real_trace_all_colon.jsonl')
print('tau2  slash:', slash[0]['actions'][0]['resource'],
      '| colon:', renotate_trace(slash[0], 'allcolon')['actions'][0]['resource'])

# Toucan — independent real source, authorized-only, MIXED native notations
!PYTHONPATH=. python map_toucan_to_chain.py --shards 1 --max-traj 200 --seed 5
toucan = load_traces('real_toucan_all.jsonl')
mixed, discarded = augment(toucan, seed=9)   # random delimiter scheme per trace, labels re-verified
write_jsonl(mixed, 'real_toucan_mixed.jsonl')
from collections import Counter
print('toucan mixed notations:', dict(Counter(t['_notation'] for t in mixed)),
      '| discarded (label changed):', discarded, '| n_traces:', len(mixed))

## Step 4 — before/after: released model vs augmented model on all held-out tests

For each model: committed synthetic test, tau2 in both notations, and the
Toucan mixed-notation recognition set. The released (non-augmented) model
should be brittle (high slash / low colon); the augmented model should hold
up across notations *and* on the independent Toucan source.

In [ ]:
import json, os
json.dump({'backends': {}}, open('results_augment.json', 'w'))
REAL = 'real_trace_all.jsonl real_trace_all_colon.jsonl real_toucan_mixed.jsonl'
# BEFORE: the released (non-augmented) model on all real held-out tests
!for tf in $REAL; do echo "=== released on $tf ==="; PYTHONPATH=. python train_verifier_reward.py --eval-checkpoint esmaeil-abedi-dev/verifier-ce-qwen2.5-0.5b --test-file $tf --merge-results results_augment.json; done
# AFTER: each augmented checkpoint on committed test + all real held-out tests
!for s in 7 8 9; do ck=ckpt_augment_seed$s; if [ -d $ck ]; then for tf in benchmark_test.jsonl $REAL; do echo "=== $ck on $tf ==="; PYTHONPATH=. python train_verifier_reward.py --eval-checkpoint $ck --test-file $tf --merge-results results_augment.json; done; fi; done

In [ ]:
import json
from stats import wilson_ci
d = json.load(open('results_augment.json'))
def pc(x): return 'n/a' if x is None else format(100*x, '5.1f')
print(f"{'checkpoint / test':58} {'n':>4} {'acc% [CI]':>20} {'f-auth%':>8} {'f-ref%':>7}")
for name in sorted(d['backends']):
    m = d['backends'][name]['metrics']; n = m['n_actions']
    c = wilson_ci(round(m['accuracy']*n), n)
    print(f"{name.replace('local:','')[:58]:58} {n:4d} "
          f"{100*m['accuracy']:5.1f} [{100*c[0]:.1f},{100*c[1]:.1f}] "
          f"{pc(m['false_authorize_rate']):>8} {pc(m['false_refuse_rate']):>7}")
print('\nKEY: released ~90% tau2-slash / ~55% tau2-colon (brittle to notation);')
print('augmented should be high on BOTH tau2 notations AND the Toucan mixed set.')
print('Toucan is authorized-only -> f-auth is n/a; its metric is f-ref (false-refuse):')
print('low f-ref = the model recognizes real legitimate calls across real formats.')

In [ ]:
!zip -j augment_results.zip training_log_augment_*.jsonl results_augment.json real_toucan_mixed.jsonl real_trace_all_colon.jsonl
from google.colab import files
files.download('augment_results.zip')